# Global Superstore Retail Dashboard Analysis

## Analysis roadmap

1. Define the retail business problem.
2. Translate it into business questions and KPIs.
3. Understand the workbook and the grain of each sheet.
4. Clean and combine the data.
5. Apply the return-status business rule.
6. Calculate and validate the KPIs.
7. Create a pandas summary for each dashboard view.
8. Create a Plotly chart from each relevant summary.
9. Reuse the prepared objects in a Dash application.

# Retail business problem

The management team of a global retail company needs a dashboard to monitor sales, profitability, order activity and returns across time, markets, regions and product categories. 

They want to identify growth trends, strong and weak business areas, and how returned orders affect performance.

### Proposed analytical solution

Create an interactive retail dashboard that presents sales, profit, orders, returns and operational performance across time, markets, regions and product categories.

## Main business questions

1. How are sales and profit changing over time?
2. Which markets, regions and product categories generate the most sales?
3. Which areas generate high sales but low or negative profit?
4. How many distinct customer orders were placed?
5. What is the average order value?
6. How much was spent on shipping?
7. How many orders were returned?
8. What percentage of orders were returned?
9. How much recorded sales and profit are associated with returned versus non-returned orders?

### How these questions address the broader problem

| Information gap | Question that addresses it | Management use |
|---|---|---|
| No clear performance trend | Sales and profit over time | Detect growth, decline and seasonal changes |
| Unclear sources of revenue | Sales by market, region and category | Identify strong and weak business areas |
| Revenue may hide poor profitability | Sales compared with profit and margin | Find high-sales, low-profit areas |
| Unclear customer-order activity | Distinct orders and average order value | Understand order volume and value |
| Unclear operational cost | Shipping cost and completion time | Review fulfilment efficiency |
| Unclear effect of returns | Returned orders, return rate and returned-order sales | Monitor possible revenue and customer-experience risk |

## Dashboard KPIs

### Headline KPIs

These are the numbers that should be visible at the top of the final dashboard:

| KPI | Definition |
|---|---|
| Retained sales | Recorded sales from orders not listed as returned |
| Retained profit | Recorded profit from orders not listed as returned |
| Retained profit margin | Retained profit divided by retained sales |
| Retained orders | Distinct non-returned order keys |
| Return rate | Returned distinct orders divided by all distinct orders |

### Supporting KPIs

- Units sold
- Average order value
- Total shipping cost
- Returned-order sales
- Returned-order profit

Not every KPI needs a large card. Headline KPIs provide the overview, while supporting KPIs can appear in secondary cards, charts, hover information or detail tables.

## Business rules and assumptions

Before calculating any KPI, we must define what its values mean.

1. **An order is identified by both market and order ID.** Some order IDs are reused in different markets, so `order_id` alone is not a globally reliable key.
2. **The Returns sheet is an order-level flag.** It does not contain a return date, returned quantity, refund amount or partial-return indicator.
3. **For this exercise, every listed return is treated as a fully returned order.** All lines belonging to that order receive `return_status = "Returned"`.
4. **Returned-order sales are not confirmed refunds.** They are the original recorded sales associated with orders listed as returned.
5. **Retained sales** means recorded sales from orders not listed in the Returns sheet.
6. **Profit margin is calculated from totals:** total profit divided by total sales. We do not average row-level percentages.

These assumptions should be shown in the final dashboard's notes or documentation. Come back to this

## Planned dashboard output

The final Dash application will use the results from this notebook.

| Dashboard area | Information |
|---|---|
| Filters | Year, market and product category |
| Headline cards | Retained sales, retained profit, profit margin, retained orders and return rate |
| Trend view | Monthly retained sales and monthly return rate |
| Performance views | Market, region, category and product performance |
| Diagnostic view | Subcategory sales compared with profit |
| Operations view | Shipping cost and order-completion time |

We are therefore “starting at the end”: every calculation and chart below has a planned place in the final dashboard.

# 1. Set up the notebook

In [5]:
# Install the required packages
%pip install pandas plotly

Note: you may need to restart the kernel to use updated packages.


In [6]:
# Import the required Packages

import pandas as pd
import plotly.express as px
from IPython.display import display

# Make tables easier to inspect
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Load the Data

In [7]:
data = ("../data/superstore_sales.xlsx")

excel_file = pd.ExcelFile(data)

excel_file.sheet_names

['Orders', 'Returns', 'People']

In [8]:
# Load Each Sheet Seprately

orders_raw = pd.read_excel(
    data,
    sheet_name="Orders"
)

returns_raw = pd.read_excel(
    data,
    sheet_name="Returns"
)

people_raw = pd.read_excel(
    data,
    sheet_name="People"
)

In [9]:
# Check their sizes

print("Orders:", orders_raw.shape)
print("Returns:", returns_raw.shape)
print("People:", people_raw.shape)

Orders: (51290, 21)
Returns: (1173, 3)
People: (14, 2)


## 3. Understand the grain

The **grain** states what one row represents.

| Sheet | Grain |
|---|---|
| Orders | One product line within an order |
| Returns | One returned order within a market |
| People | One manager-region assignment |

An order can contain several product lines. Therefore, the number of rows in `orders_raw` is not the number of customer orders.

For example, if one order contains a chair, a desk and a lamp, the Orders sheet contains three rows but only one order. This is why dashboard order counts must use a distinct order key.

In [10]:
display(orders_raw.head())
display(returns_raw.head())
display(people_raw.head())

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",408.30,2,0.00,106.14,35.46,Medium,2011
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",120.37,3,0.10,36.04,9.72,Medium,2011
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",66.12,4,0.00,29.64,8.17,High,2011
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",44.87,3,0.50,-26.06,4.82,High,2011
4,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,FUR-FU-10003447,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",113.67,5,0.10,37.77,4.70,Medium,2011


,Returned,Order ID,Market
0,Yes,MX-2013-168137,LATAM
1,Yes,US-2011-165316,LATAM
2,Yes,ES-2013-1525878,EU
3,Yes,CA-2013-118311,United States
4,Yes,ES-2011-1276768,EU


,Person,Region
0,Anna Andreadi,Central
1,Chuck Magee,South
2,Kelly Williams,East
3,Matt Collister,West
4,Deborah Brumfield,Africa


## 4. Inspect structure and data types

This first inspection helps us identify the available dimensions, measures and possible conversion problems.

- **Dimensions** describe or group the data: date, market, region, category and product.
- **Measures** are values we aggregate: sales, profit, quantity, discount and shipping cost.
- **Identifiers** such as order ID are labels, even when they look numerical. We count them; we do not add them.

In [11]:
print("Orders columns:")
print(orders_raw.columns.tolist())

print("\nReturns columns:")
print(returns_raw.columns.tolist())

print("\nOrders data types:")
orders_raw.info()

Orders columns:
['order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_name', 'segment', 'state', 'country', 'market', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'order_priority', 'year']

Returns columns:
['Returned', 'Order ID', 'Market']

Orders data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        51290 non-null  object        
 1   order_date      51290 non-null  datetime64[ns]
 2   ship_date       51290 non-null  datetime64[ns]
 3   ship_mode       51290 non-null  object        
 4   customer_name   51290 non-null  object        
 5   segment         51290 non-null  object        
 6   state           51290 non-null  object        
 7   country         51290 non-null  object        
 8   market      

# 5. Clean and standardize the data

In [12]:
def clean_column_names(dataframe):
    cleaned = dataframe.copy()
    cleaned.columns = (
        cleaned.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return cleaned


orders = clean_column_names(orders_raw)
returns = clean_column_names(returns_raw)
people = clean_column_names(people_raw)

print("Orders columns:", orders.columns.tolist())
print("Returns columns:", returns.columns.tolist())

Orders columns: ['order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_name', 'segment', 'state', 'country', 'market', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'order_priority', 'year']
Returns columns: ['returned', 'order_id', 'market']


## 5. Check missing values and exact duplicates

An exact duplicate is a row for which every column is identical. A repeated order ID is not automatically a duplicate because an order normally has several product lines.

In [13]:
quality_overview = pd.DataFrame(
    {
        "table": ["Orders", "Returns", "People"],
        "rows": [len(orders), len(returns), len(people)],
        "columns": [orders.shape[1], returns.shape[1], people.shape[1]],
        "missing_cells": [
            int(orders.isna().sum().sum()),
            int(returns.isna().sum().sum()),
            int(people.isna().sum().sum()),
        ],
        "exact_duplicate_rows": [
            int(orders.duplicated().sum()),
            int(returns.duplicated().sum()),
            int(people.duplicated().sum()),
        ],
    }
)

display(quality_overview)

,table,rows,columns,missing_cells,exact_duplicate_rows
0,Orders,51290,21,0,0
1,Returns,1173,3,0,0
2,People,14,2,0,0


## 6. Clean text and Standardize Categorical Variables

We remove leading and trailing spaces from text columns.

In [14]:
for dataframe in [orders, returns, people]:
    text_columns = dataframe.select_dtypes(include=["object", "string"]).columns
    for column in text_columns:
        dataframe[column] = dataframe[column].astype("string").str.strip()

# The Returns sheet uses "United States", while Orders uses "US".
# They must match before the merge.
returns["market"] = returns["market"].replace(
    {"United States": "US"}
)

# Standardize the return flag itself.
returns["returned"] = returns["returned"].str.title()

print("Order markets:", sorted(orders["market"].unique()))
print("Return markets:", sorted(returns["market"].unique()))

Order markets: ['APAC', 'Africa', 'Canada', 'EMEA', 'EU', 'LATAM', 'US']
Return markets: ['APAC', 'EU', 'LATAM', 'US']


## Create reliable identifiers

The Orders sheet has no line identifier, so we create `line_id` as a technical key for each order line.

More importantly, some `order_id` values are reused in different markets. We therefore create:

```text
order_key = market + "|" + order_id
```

This composite key identifies a customer order globally within this dataset.

In [15]:
orders["line_id"] = range(1, len(orders) + 1)
orders["order_key"] = (
    orders["market"] + "|" + orders["order_id"]
)
returns["order_key"] = (
    returns["market"] + "|" + returns["order_id"]
)

reused_order_ids = (
    orders[["order_id", "market"]]
    .drop_duplicates()
    .groupby("order_id", as_index=False)
    .agg(number_of_markets=("market", "nunique"))
    .query("number_of_markets > 1")
)

identifier_check = pd.DataFrame(
    {
        "measure": [
            "Order-line rows",
            "Distinct order_id values",
            "Distinct composite order_key values",
            "Order IDs reused across markets",
        ],
        "value": [
            len(orders),
            orders["order_id"].nunique(),
            orders["order_key"].nunique(),
            len(reused_order_ids),
        ],
    }
)

display(identifier_check)
display(reused_order_ids.head())

,measure,value
0,Order-line rows,51290
1,Distinct order_id values,25035
2,Distinct composite order_key values,25072
3,Order IDs reused across markets,37


,order_id,number_of_markets
22975,US-2011-130379,2
22985,US-2011-133130,2
23030,US-2011-144078,2
23102,US-2011-155894,2
23144,US-2011-160780,2


### Why we do not drop the repeated return ID

The same order ID can occur in two markets. Once market is included, every Returns record has a unique `order_key`. The repeated ID is therefore not a duplicate business record.

In [16]:
return_id_check = pd.DataFrame(
    {
        "measure": [
            "Return rows",
            "Distinct return order_id values",
            "Distinct return order_key values",
            "Duplicate composite return keys",
        ],
        "value": [
            len(returns),
            returns["order_id"].nunique(),
            returns["order_key"].nunique(),
            returns.duplicated(["order_id", "market"]).sum(),
        ],
    }
)

display(return_id_check)

,measure,value
0,Return rows,1173
1,Distinct return order_id values,1172
2,Distinct return order_key values,1173
3,Duplicate composite return keys,0


## 7.Convert dates and numerical columns

`errors="coerce"` converts an invalid value to a missing value rather than stopping the whole operation. We check for failed conversions immediately afterward.

In [17]:
date_columns = ["order_date", "ship_date"]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce",
    )

numeric_columns = [
    "sales",
    "quantity",
    "discount",
    "profit",
    "shipping_cost",
]

for column in numeric_columns:
    orders[column] = pd.to_numeric(
        orders[column],
        errors="coerce",
    )

conversion_check = pd.concat(
    [
        orders[date_columns].isna().sum(),
        orders[numeric_columns].isna().sum(),
    ]
).rename("missing_after_conversion")

display(conversion_check.to_frame())

,missing_after_conversion
order_date,0
ship_date,0
sales,0
quantity,0
discount,0
profit,0
shipping_cost,0


# 8. Add return status to every order line

The merge uses both `order_id` and `market`. `validate="many_to_one"` tells pandas that many order lines may match one Returns record, but each return key must appear no more than once.

We use a left merge so that no order line is lost.

In [18]:
return_lookup = returns[["order_id", "market", "returned"]].copy()

assert not return_lookup.duplicated(["order_id", "market"]).any()

rows_before_merge = len(orders)

df_clean = orders.merge(
    return_lookup,
    on=["order_id", "market"],
    how="left",
    validate="many_to_one",
)

assert len(df_clean) == rows_before_merge

df_clean["return_status"] = (
    df_clean["returned"]
    .map({"Yes": "Returned"})
    .fillna("Not Returned")
)

df_clean = df_clean.drop(columns="returned")

print("Order-line return status:")
display(df_clean["return_status"].value_counts().to_frame("order_lines"))

print("Distinct orders by return status:")
display(
    df_clean.groupby("return_status")["order_key"]
    .nunique()
    .to_frame("distinct_orders")
)

Order-line return status:


,order_lines
return_status,
Not Returned,48247
Returned,3043


Distinct orders by return status:


,distinct_orders
return_status,
Not Returned,23899
Returned,1173


In [ ]:
# Save clean dataset to csv
df_clean.to_csv("../data/clean_data.csv", index=False)

## 9. Create calculated measures and time dimensions

- `sales_per_unit` is recorded net sales divided by quantity. It is not necessarily the product's list price.
- `profit_margin` is useful for inspecting individual rows, but dashboard margins will be calculated from aggregated totals.
- `shipping_days` is the number of days from order date to ship date.
- `month_start` is a real date and is convenient for Plotly time axes.

In [17]:
df_clean["sales_per_unit"] = (
    df_clean["sales"]
    .div(df_clean["quantity"].where(df_clean["quantity"].ne(0)))
)

df_clean["profit_margin"] = (
    df_clean["profit"]
    .div(df_clean["sales"].where(df_clean["sales"].ne(0)))
)

df_clean["shipping_days"] = (
    df_clean["ship_date"] - df_clean["order_date"]
).dt.days

df_clean["year"] = df_clean["order_date"].dt.year
df_clean["quarter"] = df_clean["order_date"].dt.to_period("Q").astype(str)
df_clean["month_number"] = df_clean["order_date"].dt.month
df_clean["month_name"] = df_clean["order_date"].dt.month_name()
df_clean["year_month"] = df_clean["order_date"].dt.to_period("M").astype(str)
df_clean["month_start"] = df_clean["order_date"].dt.to_period("M").dt.to_timestamp()

display(
    df_clean[
        [
            "order_key",
            "order_date",
            "ship_date",
            "shipping_days",
            "year_month",
            "sales_per_unit",
            "profit_margin",
            "return_status",
        ]
    ].head()
)

,order_key,order_date,ship_date,shipping_days,year_month,sales_per_unit,profit_margin,return_status
0,Africa|AG-2011-2040,2011-01-01,2011-01-06,5,2011-01,204.15,0.26,Not Returned
1,APAC|IN-2011-47883,2011-01-01,2011-01-08,7,2011-01,40.12,0.30,Not Returned
2,EMEA|HU-2011-1220,2011-01-01,2011-01-05,4,2011-01,16.53,0.45,Not Returned
3,EU|IT-2011-3647632,2011-01-01,2011-01-05,4,2011-01,14.96,-0.58,Not Returned
4,APAC|IN-2011-47883,2011-01-01,2011-01-08,7,2011-01,22.73,0.33,Not Returned


# 10. Apply the return-status business rule

Cleaning tells us whether the data is technically usable. A business rule tells us which valid records belong in a metric.

We keep three useful DataFrames:

- `df_clean`: every order line
- `returned_sales`: order lines belonging to listed returned orders
- `retained_sales`: order lines not listed as returned

This preserves transparency. We do not silently delete returned orders.

In [18]:
returned_sales = df_clean[
    df_clean["return_status"] == "Returned"
].copy()

retained_sales = df_clean[
    df_clean["return_status"] == "Not Returned"
].copy()

print("All order lines:", f"{len(df_clean):,}")
print("Returned-order lines:", f"{len(returned_sales):,}")
print("Retained order lines:", f"{len(retained_sales):,}")

All order lines: 51,290
Returned-order lines: 3,043
Retained order lines: 48,247


# 11. Calculate dashboard KPIs

We calculate gross, returned and retained measures separately.

For the main dashboard cards, we use retained values so management can see the business remaining after orders listed as returned. Gross values remain available for reconciliation and context.

In [19]:
# Gross measures: all recorded order lines
gross_sales = df_clean["sales"].sum()
gross_profit = df_clean["profit"].sum()
gross_orders = df_clean["order_key"].nunique()
gross_units = df_clean["quantity"].sum()
gross_shipping_cost = df_clean["shipping_cost"].sum()
gross_average_order_value = gross_sales / gross_orders
gross_profit_margin = gross_profit / gross_sales

# Returned-order measures
returned_order_sales = returned_sales["sales"].sum()
returned_order_profit = returned_sales["profit"].sum()
returned_orders = returned_sales["order_key"].nunique()
returned_units = returned_sales["quantity"].sum()
returned_order_average_value = returned_order_sales / returned_orders
return_rate = returned_orders / gross_orders

# Retained dashboard measures
total_sales = retained_sales["sales"].sum()
total_profit = retained_sales["profit"].sum()
total_orders = retained_sales["order_key"].nunique()
units_sold = retained_sales["quantity"].sum()
average_order_value = total_sales / total_orders
overall_profit_margin = total_profit / total_sales
total_shipping_cost = retained_sales["shipping_cost"].sum()

kpi_summary = pd.DataFrame(
    {
        "KPI": [
            "Retained sales",
            "Retained profit",
            "Retained profit margin",
            "Retained orders",
            "Return rate",
            "Units sold",
            "Average order value",
            "Shipping cost",
            "Returned-order sales",
            "Returned-order profit",
        ],
        "Value": [
            f"${total_sales:,.2f}",
            f"${total_profit:,.2f}",
            f"{overall_profit_margin:.2%}",
            f"{total_orders:,}",
            f"{return_rate:.2%}",
            f"{units_sold:,}",
            f"${average_order_value:,.2f}",
            f"${total_shipping_cost:,.2f}",
            f"${returned_order_sales:,.2f}",
            f"${returned_order_profit:,.2f}",
        ],
    }
)

display(kpi_summary)

,KPI,Value
0,Retained sales,"$11,824,457.77"
1,Retained profit,"$1,349,289.60"
2,Retained profit margin,11.41%
3,Retained orders,"23,899"
4,Return rate,4.68%
5,Units sold,"166,567"
6,Average order value,$494.77
7,Shipping cost,"$1,263,830.72"
8,Returned-order sales,"$818,044.14"
9,Returned-order profit,"$119,745.22"


### Expected full-dataset KPI results

Use these values to confirm that your workbook and calculations match this lesson:

| KPI | Expected value |
|---|---:|
| Gross sales | $12,642,501.91 |
| Gross profit | $1,469,034.82 |
| Gross distinct orders | 25,072 |
| Retained sales | $11,824,457.77 |
| Retained profit | $1,349,289.60 |
| Retained distinct orders | 23,899 |
| Retained units | 166,567 |
| Average retained order value | $494.77 |
| Retained profit margin | 11.41% |
| Retained shipping cost | $1,263,830.72 |
| Returned distinct orders | 1,173 |
| Return rate | 4.68% |
| Returned-order sales | $818,044.14 |

Small display-rounding differences are acceptable. Large differences should be investigated before charts are built.

# 12. Create dashboard-specific summaries and Plotly charts


1. Start with a business question.
2. Aggregate the required data with pandas.
3. Validate or inspect the summary.
4. Pass the summary into Plotly Express.

### 12.1 Monthly retained performance

**Business question:** How are retained sales and profit changing over time?

One row in `monthly_summary` represents one calendar month.

In [20]:
monthly_summary = (
    retained_sales
    .groupby("month_start", as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_key", "nunique"),
        units_sold=("quantity", "sum"),
        shipping_cost=("shipping_cost", "sum"),
    )
    .sort_values("month_start")
)

monthly_summary["profit_margin"] = (
    monthly_summary["total_profit"] / monthly_summary["total_sales"]
)

display(monthly_summary.head())

,month_start,total_sales,total_profit,total_orders,units_sold,shipping_cost,profit_margin
0,2011-01-01,"96,658.74","8,170.08",211,1383,"10,397.39",0.08
1,2011-02-01,"80,144.33","8,504.04",173,1140,"9,921.66",0.11
2,2011-03-01,"142,937.33","15,091.52",271,1775,"12,766.72",0.11
3,2011-04-01,"111,917.96","13,175.23",253,1864,"12,195.46",0.12
4,2011-05-01,"138,353.62","10,862.73",281,1915,"15,580.47",0.08


In [21]:
fig_monthly_sales = px.line(
    monthly_summary,
    x="month_start",
    y="total_sales",
    markers=True,
    title="Retained Sales Over Time",
    labels={
        "month_start": "Month",
        "total_sales": "Retained Sales",
    },
    hover_data={
        "total_profit": ":$,.2f",
        "profit_margin": ":.1%",
        "total_orders": ":,",
        "units_sold": ":,",
    },
    template="plotly_white",
)

fig_monthly_sales.update_yaxes(tickprefix="$", tickformat=",")
fig_monthly_sales.update_layout(hovermode="x unified")
fig_monthly_sales.show()

**What to look for:** long-term growth or decline, recurring seasonal peaks and unusual monthly changes. The chart reveals patterns, but it does not prove what caused them.

### 12.2 Market performance

**Business question:** Which global markets generate the most retained sales, and are those sales profitable?

One row in `market_summary` represents one market.

In [22]:
market_summary = (
    retained_sales
    .groupby("market", as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_key", "nunique"),
        units_sold=("quantity", "sum"),
        shipping_cost=("shipping_cost", "sum"),
    )
)

market_summary["profit_margin"] = (
    market_summary["total_profit"] / market_summary["total_sales"]
)
market_summary["average_order_value"] = (
    market_summary["total_sales"] / market_summary["total_orders"]
)

market_summary = market_summary.sort_values(
    "total_sales",
    ascending=False,
)

display(market_summary)

,market,total_sales,total_profit,total_orders,units_sold,shipping_cost,profit_margin,average_order_value
0,APAC,"3,320,207.02","389,324.40",5141,38145,"357,405.44",0.12,645.83
4,EU,"2,733,169.55","341,211.80",4309,35237,"286,403.67",0.12,634.29
6,US,"2,116,696.58","263,164.66",4713,34820,"218,513.69",0.12,449.12
5,LATAM,"1,997,521.93","205,001.75",4841,35451,"217,587.08",0.10,412.63
3,EMEA,"806,161.31","43,897.97",2462,11517,"88,375.73",0.05,327.44
1,Africa,"783,773.21","88,871.63",2232,10564,"88,139.47",0.11,351.15
2,Canada,"66,928.17","17,817.39",201,833,"7,405.63",0.27,332.98


In [23]:
market_chart_data = market_summary.sort_values(
    "total_sales",
    ascending=True,
)

fig_market_sales = px.bar(
    market_chart_data,
    x="total_sales",
    y="market",
    orientation="h",
    color="total_sales",
    color_continuous_scale="Blues",
    title="Retained Sales by Market",
    labels={
        "market": "Market",
        "total_sales": "Retained Sales",
    },
    hover_data={
        "total_profit": ":$,.2f",
        "profit_margin": ":.1%",
        "total_orders": ":,",
        "average_order_value": ":$,.2f",
    },
    template="plotly_white",
)

fig_market_sales.update_xaxes(tickprefix="$", tickformat=",")
fig_market_sales.update_layout(coloraxis_showscale=False)
fig_market_sales.show()

**Interpret carefully:** a high profit margin on a very small sales base does not automatically make a market the company's strongest market. Review sales, profit and margin together.

### 12.3 Product-category performance

**Business question:** Which product categories generate the most retained sales and profit?

One row in `category_summary` represents one product category.

In [24]:
category_summary = (
    retained_sales
    .groupby("category", as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_key", "nunique"),
        units_sold=("quantity", "sum"),
        shipping_cost=("shipping_cost", "sum"),
    )
)

category_summary["profit_margin"] = (
    category_summary["total_profit"] / category_summary["total_sales"]
)
category_summary["average_order_value"] = (
    category_summary["total_sales"] / category_summary["total_orders"]
)

category_summary = category_summary.sort_values(
    "total_sales",
    ascending=False,
)

display(category_summary)

,category,total_sales,total_profit,total_orders,units_sold,shipping_cost,profit_margin,average_order_value
2,Technology,"4,438,054.34","609,638.27",7873,32815,"475,991.48",0.14,563.71
0,Furniture,"3,846,499.60","262,363.46",7720,32517,"409,474.74",0.07,498.25
1,Office Supplies,"3,539,903.83","477,287.87",18083,101235,"378,364.50",0.13,195.76


> Category order counts are **non-additive**. One customer order can contain products from several categories, so adding the category order counts can double-count orders. Use `total_orders` from the KPI section for the overall order count.

In [25]:
fig_category_sales = px.bar(
    category_summary,
    x="category",
    y="total_sales",
    color="category",
    title="Retained Sales by Product Category",
    labels={
        "category": "Product Category",
        "total_sales": "Retained Sales",
    },
    hover_data={
        "total_profit": ":$,.2f",
        "profit_margin": ":.1%",
        "total_orders": ":,",
        "units_sold": ":,",
    },
    template="plotly_white",
)

fig_category_sales.update_yaxes(tickprefix="$", tickformat=",")
fig_category_sales.update_layout(showlegend=False)
fig_category_sales.show()

### 12.4 Market-region performance

**Business question:** Which regions within each market generate the most retained sales?

Region names may not be globally unique, so we group by both market and region and create a readable combined label.

In [26]:
region_summary = (
    retained_sales
    .groupby(["market", "region"], as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_key", "nunique"),
        units_sold=("quantity", "sum"),
        shipping_cost=("shipping_cost", "sum"),
    )
)

region_summary["profit_margin"] = (
    region_summary["total_profit"] / region_summary["total_sales"]
)
region_summary["average_order_value"] = (
    region_summary["total_sales"] / region_summary["total_orders"]
)
region_summary["market_region"] = (
    region_summary["market"] + " — " + region_summary["region"]
)

region_summary = region_summary.sort_values(
    "total_sales",
    ascending=False,
)

display(region_summary.head(15))

,market,region,total_sales,total_profit,total_orders,units_sold,shipping_cost,profit_margin,average_order_value,market_region
7,EU,Central,"1,613,021.82","199,498.02",2413,20658,"172,062.11",0.12,668.47,EU — Central
2,APAC,Oceania,"1,055,576.98","113,672.98",1686,12324,"115,566.73",0.11,626.08,APAC — Oceania
3,APAC,Southeast Asia,"840,744.77","17,519.19",1462,11187,"89,133.09",0.02,575.06,APAC — Southeast Asia
6,EMEA,EMEA,"806,161.31","43,897.97",2462,11517,"88,375.73",0.05,327.44,EMEA — EMEA
4,Africa,Africa,"783,773.21","88,871.63",2232,10564,"88,139.47",0.11,351.15,Africa — Africa
0,APAC,Central Asia,"729,142.43","127,787.88",1002,7451,"74,585.38",0.18,727.69,APAC — Central Asia
1,APAC,North Asia,"694,742.84","130,344.35",991,7183,"78,120.24",0.19,701.05,APAC — North Asia
15,US,East,"637,076.10","86,537.98",1357,10075,"67,699.83",0.14,469.47,US — East
17,US,West,"617,974.77","88,755.07",1422,10368,"65,776.52",0.14,434.58,US — West
13,LATAM,South,"590,809.82","28,495.78",1407,10746,"64,846.96",0.05,419.91,LATAM — South


In [27]:
top_regions = (
    region_summary
    .nlargest(15, "total_sales")
    .sort_values("total_sales", ascending=True)
)

fig_region_sales = px.bar(
    top_regions,
    x="total_sales",
    y="market_region",
    orientation="h",
    color="market",
    title="Top 15 Market-Regions by Retained Sales",
    labels={
        "market_region": "Market — Region",
        "total_sales": "Retained Sales",
        "market": "Market",
    },
    hover_data={
        "total_profit": ":$,.2f",
        "profit_margin": ":.1%",
        "total_orders": ":,",
        "units_sold": ":,",
    },
    template="plotly_white",
)

fig_region_sales.update_xaxes(tickprefix="$", tickformat=",")
fig_region_sales.update_layout(margin={"l": 160})
fig_region_sales.show()

### 12.5 Subcategory sales and profitability

**Business question:** Which subcategories generate high sales but weak or negative profit?

A scatter plot is appropriate because it compares two numerical measures. Points to the right have higher sales; points higher on the chart have higher profit. Points below the zero-profit line are loss-making in aggregate.

In [28]:
subcategory_summary = (
    retained_sales
    .groupby(["category", "sub_category"], as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_key", "nunique"),
        units_sold=("quantity", "sum"),
        average_discount=("discount", "mean"),
        shipping_cost=("shipping_cost", "sum"),
    )
)

subcategory_summary["profit_margin"] = (
    subcategory_summary["total_profit"]
    / subcategory_summary["total_sales"]
)

subcategory_summary = subcategory_summary.sort_values(
    "total_sales",
    ascending=False,
)

display(subcategory_summary)

,category,sub_category,total_sales,total_profit,total_orders,units_sold,average_discount,shipping_cost,profit_margin
16,Technology,Phones,"1,606,667.20","201,440.82",2949,11119,0.15,"174,852.66",0.13
1,Furniture,Chairs,"1,411,625.67","131,557.51",2997,11500,0.16,"152,384.18",0.09
14,Technology,Copiers,"1,398,970.94","234,604.16",1994,6963,0.12,"149,369.09",0.17
0,Furniture,Bookcases,"1,374,492.65","148,808.13",2149,7760,0.16,"144,945.86",0.11
11,Office Supplies,Storage,"1,065,400.12","102,445.33",4317,15968,0.14,"113,258.69",0.10
4,Office Supplies,Appliances,"923,627.63","123,036.81",1566,5567,0.14,"99,608.40",0.13
15,Technology,Machines,"736,690.96","55,492.73",1353,4619,0.17,"74,005.91",0.08
3,Furniture,Tables,"704,798.16","-60,060.55",773,2819,0.29,"74,377.33",-0.09
13,Technology,Accessories,"695,725.24","118,100.56",2689,10114,0.12,"77,763.81",0.17
6,Office Supplies,Binders,"438,345.79","71,908.18",5101,20011,0.18,"45,514.70",0.16


In [29]:
fig_subcategory_profit = px.scatter(
    subcategory_summary,
    x="total_sales",
    y="total_profit",
    color="category",
    size="units_sold",
    hover_name="sub_category",
    hover_data={
        "profit_margin": ":.1%",
        "average_discount": ":.1%",
        "total_orders": ":,",
        "units_sold": ":,",
    },
    title="Subcategory Sales and Profit",
    labels={
        "total_sales": "Retained Sales",
        "total_profit": "Retained Profit",
        "category": "Product Category",
        "units_sold": "Units Sold",
    },
    template="plotly_white",
)

fig_subcategory_profit.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
)
fig_subcategory_profit.update_xaxes(tickprefix="$", tickformat=",")
fig_subcategory_profit.update_yaxes(tickprefix="$", tickformat=",")
fig_subcategory_profit.show()

**What to investigate:** subcategories with large sales but small, zero or negative profit. Discounting, product cost, shipping and market mix may be possible explanations, but the chart alone does not establish causation.

### 12.6 Top products

**Business question:** Which individual products generate the most retained sales?

The full product table remains in `product_summary`. The chart displays only the top ten so that product names stay readable.

In [30]:
product_summary = (
    retained_sales
    .groupby(
        ["product_id", "product_name", "category", "sub_category"],
        as_index=False,
    )
    .agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        total_orders=("order_key", "nunique"),
        units_sold=("quantity", "sum"),
    )
)

product_summary["profit_margin"] = (
    product_summary["total_profit"] / product_summary["total_sales"]
)

product_summary = product_summary.sort_values(
    "total_sales",
    ascending=False,
)

top_products = (
    product_summary
    .nlargest(10, "total_sales")
    .sort_values("total_sales", ascending=True)
)

display(top_products)

,product_id,product_name,category,sub_category,total_sales,total_profit,total_orders,units_sold,profit_margin
9037,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,"18,839.69","6,983.88",8,38,0.37
3602,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,Binders,"19,024.50",760.98,9,48,0.04
741,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,"19,417.15",350.49,7,34,0.02
3711,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,"19,823.48","2,233.51",11,37,0.11
538,FUR-CH-10000027,"SAFCO Executive Leather Armchair, Black",Furniture,Chairs,"21,329.73","1,363.23",12,50,0.06
10566,TEC-PH-10004823,"Nokia Smart Phone, Full Size",Technology,Phones,"22,262.10","8,121.48",11,39,0.36
9650,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,"22,638.48","-1,811.08",1,6,-0.08
3984,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,"27,453.38","7,753.04",10,31,0.28
10552,TEC-PH-10004664,"Nokia Smart Phone, with Caller ID",Technology,Phones,"30,041.55","5,455.95",10,52,0.18
9321,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,"47,599.86","18,479.95",4,16,0.39


In [31]:
fig_top_products = px.bar(
    top_products,
    x="total_sales",
    y="product_name",
    orientation="h",
    color="category",
    title="Top 10 Products by Retained Sales",
    labels={
        "product_name": "Product",
        "total_sales": "Retained Sales",
        "category": "Product Category",
    },
    hover_data={
        "product_id": True,
        "sub_category": True,
        "total_profit": ":$,.2f",
        "profit_margin": ":.1%",
        "total_orders": ":,",
        "units_sold": ":,",
    },
    template="plotly_white",
)

fig_top_products.update_xaxes(tickprefix="$", tickformat=",")
fig_top_products.update_layout(margin={"l": 260})
fig_top_products.show()

## 13. Build an order-level table for return and shipping analysis

The Orders sheet has an order-line grain. Return rate, however, is an order-level metric. We therefore create a separate `order_level` table with one row per composite order key.

Before aggregating, we confirm that every line belonging to the same order has one consistent return status.

In [32]:
status_counts_per_order = (
    df_clean.groupby("order_key")["return_status"].nunique()
)
assert status_counts_per_order.eq(1).all()

order_level = (
    df_clean
    .groupby(["order_key", "market"], as_index=False)
    .agg(
        order_id=("order_id", "first"),
        order_date=("order_date", "min"),
        order_sales=("sales", "sum"),
        order_profit=("profit", "sum"),
        order_units=("quantity", "sum"),
        order_shipping_cost=("shipping_cost", "sum"),
        order_completion_days=("shipping_days", "max"),
        return_status=("return_status", "first"),
    )
)

order_level["month_start"] = (
    order_level["order_date"].dt.to_period("M").dt.to_timestamp()
)
order_level["returned_order"] = (
    order_level["return_status"].eq("Returned").astype(int)
)
order_level["returned_order_sales"] = (
    order_level["order_sales"]
    .where(order_level["return_status"].eq("Returned"), 0)
)

print("Order-level rows:", f"{len(order_level):,}")
display(order_level.head())

Order-level rows: 25,072


,order_key,market,order_id,order_date,order_sales,order_profit,order_units,order_shipping_cost,order_completion_days,return_status,month_start,returned_order,returned_order_sales
0,APAC|ID-2011-10706,APAC,ID-2011-10706,2011-04-18,730.01,-38.95,9,57.02,4,Not Returned,2011-04-01,0,0.00
1,APAC|ID-2011-11126,APAC,ID-2011-11126,2011-11-25,132.75,-11.31,9,8.48,4,Not Returned,2011-11-01,0,0.00
2,APAC|ID-2011-11385,APAC,ID-2011-11385,2011-08-05,60.00,-52.80,5,3.99,6,Not Returned,2011-08-01,0,0.00
3,APAC|ID-2011-11392,APAC,ID-2011-11392,2011-09-05,84.96,-75.12,14,21.06,2,Not Returned,2011-09-01,0,0.00
4,APAC|ID-2011-12596,APAC,ID-2011-12596,2011-01-03,135.12,-45.90,2,7.74,5,Not Returned,2011-01-01,0,0.00


### 13.1 Return performance by market

**Business question:** Which markets have the highest percentage of returned orders?

Markets with no listed returns remain in the summary with a zero return rate.

In [33]:
market_return_summary = (
    order_level
    .groupby("market", as_index=False)
    .agg(
        total_orders=("order_key", "nunique"),
        returned_orders=("returned_order", "sum"),
        returned_order_sales=("returned_order_sales", "sum"),
    )
)

market_return_summary["return_rate"] = (
    market_return_summary["returned_orders"]
    / market_return_summary["total_orders"]
)

market_return_summary = market_return_summary.sort_values(
    "return_rate",
    ascending=False,
)

display(market_return_summary)

,market,total_orders,returned_orders,returned_order_sales,return_rate
4,EU,4593,284,"204,919.51",0.06
6,US,5009,296,"180,504.28",0.06
5,LATAM,5138,297,"167,083.24",0.06
0,APAC,5437,296,"265,537.11",0.05
1,Africa,2232,0,0.00,0.00
2,Canada,201,0,0.00,0.00
3,EMEA,2462,0,0.00,0.00


In [34]:
fig_market_returns = px.bar(
    market_return_summary,
    x="market",
    y="return_rate",
    color="return_rate",
    color_continuous_scale="Oranges",
    title="Order Return Rate by Market",
    labels={
        "market": "Market",
        "return_rate": "Return Rate",
    },
    hover_data={
        "total_orders": ":,",
        "returned_orders": ":,",
        "returned_order_sales": ":$,.2f",
    },
    template="plotly_white",
)

fig_market_returns.update_yaxes(tickformat=".1%")
fig_market_returns.update_layout(coloraxis_showscale=False)
fig_market_returns.show()

### 13.2 Monthly return-rate trend

**Business question:** Is the percentage of returned orders changing over time?

Return rate uses distinct orders—not order-line rows—as its denominator.

In [35]:
monthly_return_summary = (
    order_level
    .groupby("month_start", as_index=False)
    .agg(
        total_orders=("order_key", "nunique"),
        returned_orders=("returned_order", "sum"),
        returned_order_sales=("returned_order_sales", "sum"),
    )
    .sort_values("month_start")
)

monthly_return_summary["return_rate"] = (
    monthly_return_summary["returned_orders"]
    / monthly_return_summary["total_orders"]
)

display(monthly_return_summary.head())

,month_start,total_orders,returned_orders,returned_order_sales,return_rate
0,2011-01-01,216,5,"4,060.68",0.02
1,2011-02-01,180,10,"11,007.83",0.06
2,2011-03-01,277,6,"2,792.04",0.02
3,2011-04-01,264,14,"4,997.81",0.05
4,2011-05-01,295,15,"8,394.21",0.05


In [36]:
fig_monthly_returns = px.line(
    monthly_return_summary,
    x="month_start",
    y="return_rate",
    markers=True,
    title="Monthly Order Return Rate",
    labels={
        "month_start": "Month",
        "return_rate": "Return Rate",
    },
    hover_data={
        "total_orders": ":,",
        "returned_orders": ":,",
        "returned_order_sales": ":$,.2f",
    },
    template="plotly_white",
)

fig_monthly_returns.update_yaxes(tickformat=".1%")
fig_monthly_returns.update_layout(hovermode="x unified")
fig_monthly_returns.show()

### 13.3 Return association by product category

**Business question:** What percentage of orders containing each category were returned?

We first create one row per order-category combination. This prevents an order with several lines from the same category from being counted repeatedly.

> This analysis shows association, not cause. If a returned order contained Furniture and Technology, it is counted once under each category. The data does not identify which product caused the return.

In [37]:
order_category = (
    df_clean
    .groupby(["order_key", "category"], as_index=False)
    .agg(
        category_sales=("sales", "sum"),
        return_status=("return_status", "first"),
    )
)

order_category["returned_order"] = (
    order_category["return_status"].eq("Returned").astype(int)
)
order_category["returned_order_sales"] = (
    order_category["category_sales"]
    .where(order_category["return_status"].eq("Returned"), 0)
)

category_return_summary = (
    order_category
    .groupby("category", as_index=False)
    .agg(
        orders_with_category=("order_key", "nunique"),
        returned_orders=("returned_order", "sum"),
        returned_order_sales=("returned_order_sales", "sum"),
    )
)

category_return_summary["return_rate"] = (
    category_return_summary["returned_orders"]
    / category_return_summary["orders_with_category"]
)

category_return_summary = category_return_summary.sort_values(
    "return_rate",
    ascending=False,
)

display(category_return_summary)

,category,orders_with_category,returned_orders,returned_order_sales,return_rate
0,Furniture,8204,484,"264,374.59",0.06
2,Technology,8358,485,"306,503.16",0.06
1,Office Supplies,19023,940,"247,166.39",0.05


In [38]:
fig_category_returns = px.bar(
    category_return_summary,
    x="category",
    y="return_rate",
    color="category",
    title="Return Rate of Orders Containing Each Category",
    labels={
        "category": "Product Category",
        "return_rate": "Return Rate",
    },
    hover_data={
        "orders_with_category": ":,",
        "returned_orders": ":,",
        "returned_order_sales": ":$,.2f",
    },
    template="plotly_white",
)

fig_category_returns.update_yaxes(tickformat=".1%")
fig_category_returns.update_layout(showlegend=False)
fig_category_returns.show()

### 13.4 Shipping efficiency by market

**Business question:** Which markets combine high shipping-cost share with long order-completion times?

We use retained orders so that the operations view is consistent with the main sales and profit KPIs.

In [39]:
retained_order_level = order_level[
    order_level["return_status"] == "Not Returned"
].copy()

shipping_summary = (
    retained_order_level
    .groupby("market", as_index=False)
    .agg(
        total_orders=("order_key", "nunique"),
        total_sales=("order_sales", "sum"),
        total_shipping_cost=("order_shipping_cost", "sum"),
        average_shipping_cost_per_order=("order_shipping_cost", "mean"),
        average_completion_days=("order_completion_days", "mean"),
    )
)

shipping_summary["shipping_cost_share"] = (
    shipping_summary["total_shipping_cost"]
    / shipping_summary["total_sales"]
)

shipping_summary = shipping_summary.sort_values(
    "shipping_cost_share",
    ascending=False,
)

display(shipping_summary)

,market,total_orders,total_sales,total_shipping_cost,average_shipping_cost_per_order,average_completion_days,shipping_cost_share
1,Africa,2232,"783,773.21","88,139.47",39.49,3.98,0.11
2,Canada,201,"66,928.17","7,405.63",36.84,3.69,0.11
3,EMEA,2462,"806,161.31","88,375.73",35.90,3.99,0.11
5,LATAM,4841,"1,997,521.93","217,587.08",44.95,3.99,0.11
0,APAC,5141,"3,320,207.02","357,405.44",69.52,3.97,0.11
4,EU,4309,"2,733,169.55","286,403.67",66.47,4.10,0.10
6,US,4713,"2,116,696.58","218,513.69",46.36,3.96,0.10


In [40]:
fig_shipping = px.scatter(
    shipping_summary,
    x="average_completion_days",
    y="shipping_cost_share",
    size="total_shipping_cost",
    color="market",
    hover_name="market",
    hover_data={
        "total_orders": ":,",
        "total_sales": ":$,.2f",
        "total_shipping_cost": ":$,.2f",
        "average_shipping_cost_per_order": ":$,.2f",
    },
    title="Shipping Efficiency by Market",
    labels={
        "average_completion_days": "Average Order Completion Time (Days)",
        "shipping_cost_share": "Shipping Cost as a Share of Sales",
        "total_shipping_cost": "Total Shipping Cost",
    },
    template="plotly_white",
)

fig_shipping.update_yaxes(tickformat=".1%")
fig_shipping.show()

**Interpret carefully:** an expensive or slow market may face geographic and service-level constraints. The chart identifies where to investigate; it does not prove operational failure.

## 14. Validate the prepared data and summaries

Assertions act as safety checks. If an assertion fails, stop and investigate before using the figures in the dashboard.

In [41]:
def values_match(first_value, second_value, tolerance=0.01):
    # Compare monetary totals while allowing a one-cent rounding tolerance.
    return abs(first_value - second_value) <= tolerance


# Row and key integrity
assert len(df_clean) == len(orders)
assert df_clean["line_id"].is_unique
assert returns["order_key"].is_unique
assert order_level["order_key"].is_unique

# Required values and valid ranges
assert df_clean[["order_date", "ship_date"]].notna().all().all()
assert df_clean[numeric_columns].notna().all().all()
assert (df_clean["quantity"] > 0).all()
assert (df_clean["sales"] >= 0).all()
assert (df_clean["shipping_cost"] >= 0).all()
assert df_clean["discount"].between(0, 1).all()
assert (df_clean["ship_date"] >= df_clean["order_date"]).all()
assert set(df_clean["return_status"].unique()) == {
    "Returned",
    "Not Returned",
}

# Every return key must exist in Orders.
return_keys = set(returns["order_key"])
order_keys = set(df_clean["order_key"])
assert return_keys.issubset(order_keys)

# KPI reconciliations
assert values_match(gross_sales, total_sales + returned_order_sales)
assert values_match(gross_profit, total_profit + returned_order_profit)
assert gross_orders == total_orders + returned_orders

# Summary reconciliations
for summary in [
    monthly_summary,
    market_summary,
    region_summary,
    category_summary,
    subcategory_summary,
    product_summary,
]:
    assert values_match(summary["total_sales"].sum(), total_sales)
    assert values_match(summary["total_profit"].sum(), total_profit)

assert values_match(market_summary["shipping_cost"].sum(), total_shipping_cost)
assert values_match(region_summary["shipping_cost"].sum(), total_shipping_cost)
assert market_return_summary["returned_orders"].sum() == returned_orders
assert values_match(
    market_return_summary["returned_order_sales"].sum(),
    returned_order_sales,
)

print("All data-quality and summary checks passed.")

All data-quality and summary checks passed.


# 11. Objects ready for the Dash application

The notebook has prepared both data and figures. The most important reusable objects are:

| Purpose | pandas object | Plotly figure |
|---|---|---|
| KPI cards | KPI variables and `kpi_summary` | Not required; Dash cards will display the values |
| Monthly sales trend | `monthly_summary` | `fig_monthly_sales` |
| Market performance | `market_summary` | `fig_market_sales` |
| Category performance | `category_summary` | `fig_category_sales` |
| Regional ranking | `region_summary` | `fig_region_sales` |
| Profit diagnostic | `subcategory_summary` | `fig_subcategory_profit` |
| Product ranking | `product_summary` | `fig_top_products` |
| Market return rate | `market_return_summary` | `fig_market_returns` |
| Monthly return rate | `monthly_return_summary` | `fig_monthly_returns` |
| Category return association | `category_return_summary` | `fig_category_returns` |
| Shipping operations | `shipping_summary` | `fig_shipping` |

In Dash, a figure will be placed in a graph component like this:

```python
dcc.Graph(
    id="monthly-sales-chart",
    figure=fig_monthly_sales,
)
```

## Recommended first dashboard page

- Five headline KPI cards
- Monthly retained-sales trend
- Retained sales by market
- Retained sales by category
- Market return-rate chart
- Subcategory sales-versus-profit diagnostic
- Shipping-efficiency chart

The region and product views can appear lower on the page or in a detail section.

## Next step

Build the Dash layout, place these figure objects into `dcc.Graph` components and add clear filters. After the static layout works, callbacks can recalculate the KPIs, summaries and figures from filtered data.

# Build the Dash dashboard

We now have everything required to build the application:

- `df_clean` contains the clean data.
- The KPI formulas have been tested.
- The pandas summaries establish the correct calculations.
- The Plotly figures establish the visual design.

Dash will connect these pieces into an interactive web application.

## The role of the layout and callbacks

| Dash concept | Responsibility in our application |
|---|---|
| Layout | Describes what appears on the page and how it is arranged |
| Component ID | Gives Dash a unique way to identify a card, filter or chart |
| Input | A property whose change triggers a callback, such as a dropdown's `value` |
| Output | A property updated by a callback, such as a card's `children` or graph's `figure` |
| Callback function | Filters the data, recalculates results and returns updated values |

The data flow is:

```text
Dropdown values → callback → filtered data → KPIs and summaries → updated dashboard
```

The callback will read the global `df_clean` DataFrame but will never modify it. Every interaction creates a filtered copy for that user's request.

## 1. Install and import Dash

In [ ]:
# Run this only if Dash is missing.
%pip install "dash>=2.11"

In [44]:
from dash import Dash, Input, Output, dcc, html
import plotly.graph_objects as go

## 2. Define the dashboard's visual style

We use plain Dash and inline CSS so that the notebook does not depend on an external theme. Keeping the colours and repeated styles in dictionaries makes the layout easier to maintain.

In [45]:
COLORS = {
    "page": "#F4F7FB",
    "surface": "#FFFFFF",
    "primary": "#1F4E78",
    "secondary": "#2F75B5",
    "accent": "#E67E22",
    "success": "#1E8449",
    "danger": "#C0392B",
    "text": "#1F2937",
    "muted": "#6B7280",
    "border": "#DDE3EA",
}

CARD_STYLE = {
    "backgroundColor": COLORS["surface"],
    "border": f"1px solid {COLORS['border']}",
    "borderRadius": "12px",
    "padding": "18px",
    "boxShadow": "0 2px 8px rgba(31, 41, 55, 0.06)",
}

GRAPH_CARD_STYLE = {
    **CARD_STYLE,
    "padding": "8px 12px 4px 12px",
    "minWidth": 0,
}

FILTER_STYLE = {
    "minWidth": "210px",
    "flex": "1 1 210px",
}

## 3. Create reusable layout components

The function below produces one KPI card. Reusing it keeps the five cards visually consistent and reduces repeated layout code.

In [46]:
def make_kpi_card(title, value_id, description, accent_color):
    return html.Div(
        [
            html.Div(
                style={
                    "width": "42px",
                    "height": "4px",
                    "backgroundColor": accent_color,
                    "borderRadius": "4px",
                    "marginBottom": "14px",
                }
            ),
            html.P(
                title,
                style={
                    "margin": "0 0 8px 0",
                    "color": COLORS["muted"],
                    "fontSize": "14px",
                    "fontWeight": "600",
                },
            ),
            html.H3(
                id=value_id,
                style={
                    "margin": "0",
                    "color": COLORS["text"],
                    "fontSize": "26px",
                },
            ),
            html.P(
                description,
                style={
                    "margin": "8px 0 0 0",
                    "color": COLORS["muted"],
                    "fontSize": "12px",
                    "lineHeight": "1.4",
                },
            ),
        ],
        style=CARD_STYLE,
    )

# 4. Create functions for filtering, KPIs and figures

The callback should remain easy to read. We therefore place repeated calculations inside small helper functions.

This separation gives each function one clear job:

- `filter_dashboard_data()` applies the selected filters.
- `calculate_dashboard_kpis()` calculates the card values.
- The `build_..._figure()` functions create one chart each.

In [47]:
def filter_dashboard_data(
    dataframe,
    selected_year="All",
    selected_market="All",
    selected_category="All",
):
    # Make a copy so the callback never changes the original df_clean.
    filtered = dataframe.copy()

    if selected_year not in [None, "All"]:
        filtered = filtered[filtered["year"] == int(selected_year)]

    if selected_market not in [None, "All"]:
        filtered = filtered[filtered["market"] == selected_market]

    if selected_category not in [None, "All"]:
        filtered = filtered[filtered["category"] == selected_category]

    return filtered


def safe_divide(numerator, denominator):
    # Return zero when a filtered selection has no valid denominator.
    return numerator / denominator if denominator else 0


def format_currency(value):
    return f"${value:,.2f}"


def format_integer(value):
    return f"{value:,.0f}"


def format_percentage(value):
    return f"{value:.2%}"

In [48]:
def calculate_dashboard_kpis(filtered_data):
    returned_data = filtered_data[
        filtered_data["return_status"] == "Returned"
    ]
    retained_data = filtered_data[
        filtered_data["return_status"] == "Not Returned"
    ]

    gross_order_count = filtered_data["order_key"].nunique()
    returned_order_count = returned_data["order_key"].nunique()
    retained_order_count = retained_data["order_key"].nunique()

    retained_sales_value = retained_data["sales"].sum()
    retained_profit_value = retained_data["profit"].sum()

    return {
        "retained_sales": retained_sales_value,
        "retained_profit": retained_profit_value,
        "profit_margin": safe_divide(
            retained_profit_value,
            retained_sales_value,
        ),
        "retained_orders": retained_order_count,
        "return_rate": safe_divide(
            returned_order_count,
            gross_order_count,
        ),
        "gross_orders": gross_order_count,
        "returned_orders": returned_order_count,
    }

## 5. Empty-chart handling and common formatting

A valid combination of filters may occasionally return no rows. Instead of showing an error, the dashboard will display a clear empty-state message.

In [49]:
def empty_figure(title):
    figure = go.Figure()
    figure.add_annotation(
        text="No data is available for this filter combination.",
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
        font={"size": 15, "color": COLORS["muted"]},
    )
    figure.update_layout(
        title=title,
        template="plotly_white",
        height=390,
        margin={"l": 50, "r": 30, "t": 65, "b": 50},
        xaxis={"visible": False},
        yaxis={"visible": False},
    )
    return figure


def finish_dashboard_figure(figure):
    figure.update_layout(
        height=390,
        margin={"l": 55, "r": 30, "t": 65, "b": 55},
        title={"x": 0.02, "xanchor": "left"},
        font={"color": COLORS["text"]},
        transition_duration=250,
    )
    return figure

## 6. Figure functions: sales performance

Each function accepts the filtered order-line data, creates the required pandas summary and returns one Plotly figure.

In [50]:
def build_monthly_sales_figure(filtered_data):
    title = "Retained Sales Over Time"
    retained = filtered_data[
        filtered_data["return_status"] == "Not Returned"
    ]

    if retained.empty:
        return empty_figure(title)

    summary = (
        retained
        .groupby("month_start", as_index=False)
        .agg(
            total_sales=("sales", "sum"),
            total_profit=("profit", "sum"),
            total_orders=("order_key", "nunique"),
        )
        .sort_values("month_start")
    )

    figure = px.line(
        summary,
        x="month_start",
        y="total_sales",
        markers=True,
        title=title,
        labels={
            "month_start": "Month",
            "total_sales": "Retained Sales",
        },
        hover_data={
            "total_profit": ":$,.2f",
            "total_orders": ":,",
        },
        template="plotly_white",
        color_discrete_sequence=[COLORS["secondary"]],
    )
    figure.update_yaxes(tickprefix="$", tickformat=",")
    figure.update_layout(hovermode="x unified")
    return finish_dashboard_figure(figure)


def build_market_sales_figure(filtered_data):
    title = "Retained Sales by Market"
    retained = filtered_data[
        filtered_data["return_status"] == "Not Returned"
    ]

    if retained.empty:
        return empty_figure(title)

    summary = (
        retained
        .groupby("market", as_index=False)
        .agg(
            total_sales=("sales", "sum"),
            total_profit=("profit", "sum"),
            total_orders=("order_key", "nunique"),
        )
    )
    summary["profit_margin"] = (
        summary["total_profit"] / summary["total_sales"]
    )
    summary = summary.sort_values("total_sales", ascending=True)

    figure = px.bar(
        summary,
        x="total_sales",
        y="market",
        orientation="h",
        color="total_sales",
        color_continuous_scale="Blues",
        title=title,
        labels={
            "market": "Market",
            "total_sales": "Retained Sales",
        },
        hover_data={
            "total_profit": ":$,.2f",
            "profit_margin": ":.1%",
            "total_orders": ":,",
        },
        template="plotly_white",
    )
    figure.update_xaxes(tickprefix="$", tickformat=",")
    figure.update_layout(coloraxis_showscale=False)
    return finish_dashboard_figure(figure)


def build_category_sales_figure(filtered_data):
    title = "Retained Sales by Product Category"
    retained = filtered_data[
        filtered_data["return_status"] == "Not Returned"
    ]

    if retained.empty:
        return empty_figure(title)

    summary = (
        retained
        .groupby("category", as_index=False)
        .agg(
            total_sales=("sales", "sum"),
            total_profit=("profit", "sum"),
            total_orders=("order_key", "nunique"),
        )
        .sort_values("total_sales", ascending=False)
    )
    summary["profit_margin"] = (
        summary["total_profit"] / summary["total_sales"]
    )

    figure = px.bar(
        summary,
        x="category",
        y="total_sales",
        color="category",
        title=title,
        labels={
            "category": "Product Category",
            "total_sales": "Retained Sales",
        },
        hover_data={
            "total_profit": ":$,.2f",
            "profit_margin": ":.1%",
            "total_orders": ":,",
        },
        template="plotly_white",
    )
    figure.update_yaxes(tickprefix="$", tickformat=",")
    figure.update_layout(showlegend=False)
    return finish_dashboard_figure(figure)

## 7. Figure functions: returns and profitability

In [51]:
def build_market_return_figure(filtered_data):
    title = "Order Return Rate by Market"

    if filtered_data.empty:
        return empty_figure(title)

    # Create one row per order before calculating the return rate.
    orders_for_chart = (
        filtered_data
        .groupby(["order_key", "market"], as_index=False)
        .agg(return_status=("return_status", "first"))
    )
    orders_for_chart["returned_order"] = (
        orders_for_chart["return_status"].eq("Returned").astype(int)
    )

    summary = (
        orders_for_chart
        .groupby("market", as_index=False)
        .agg(
            total_orders=("order_key", "nunique"),
            returned_orders=("returned_order", "sum"),
        )
    )
    summary["return_rate"] = (
        summary["returned_orders"] / summary["total_orders"]
    )
    summary = summary.sort_values("return_rate", ascending=False)

    figure = px.bar(
        summary,
        x="market",
        y="return_rate",
        color="market",
        title=title,
        labels={
            "market": "Market",
            "return_rate": "Return Rate",
        },
        hover_data={
            "total_orders": ":,",
            "returned_orders": ":,",
        },
        template="plotly_white",
    )
    figure.update_yaxes(tickformat=".1%")
    figure.update_layout(showlegend=False)
    return finish_dashboard_figure(figure)


def build_subcategory_profit_figure(filtered_data):
    title = "Subcategory Sales and Profit"
    retained = filtered_data[
        filtered_data["return_status"] == "Not Returned"
    ]

    if retained.empty:
        return empty_figure(title)

    summary = (
        retained
        .groupby(["category", "sub_category"], as_index=False)
        .agg(
            total_sales=("sales", "sum"),
            total_profit=("profit", "sum"),
            units_sold=("quantity", "sum"),
            average_discount=("discount", "mean"),
        )
    )
    summary["profit_margin"] = (
        summary["total_profit"] / summary["total_sales"]
    )

    figure = px.scatter(
        summary,
        x="total_sales",
        y="total_profit",
        color="category",
        size="units_sold",
        hover_name="sub_category",
        hover_data={
            "profit_margin": ":.1%",
            "average_discount": ":.1%",
            "units_sold": ":,",
        },
        title=title,
        labels={
            "total_sales": "Retained Sales",
            "total_profit": "Retained Profit",
            "category": "Product Category",
        },
        template="plotly_white",
    )
    figure.add_hline(
        y=0,
        line_dash="dash",
        line_color=COLORS["muted"],
    )
    figure.update_xaxes(tickprefix="$", tickformat=",")
    figure.update_yaxes(tickprefix="$", tickformat=",")
    return finish_dashboard_figure(figure)

## 8. Figure function: shipping operations

In [53]:
def build_shipping_figure(filtered_data):
    title = "Shipping Efficiency by Market"
    retained = filtered_data[
        filtered_data["return_status"] == "Not Returned"
    ]

    if retained.empty:
        return empty_figure(title)

    # Move from order-line grain to one row per retained order.
    filtered_orders = (
        retained
        .groupby(["order_key", "market"], as_index=False)
        .agg(
            order_sales=("sales", "sum"),
            order_shipping_cost=("shipping_cost", "sum"),
            order_completion_days=("shipping_days", "max"),
        )
    )

    summary = (
        filtered_orders
        .groupby("market", as_index=False)
        .agg(
            total_orders=("order_key", "nunique"),
            total_sales=("order_sales", "sum"),
            total_shipping_cost=("order_shipping_cost", "sum"),
            average_shipping_cost_per_order=(
                "order_shipping_cost",
                "mean",
            ),
            average_completion_days=("order_completion_days", "mean"),
        )
    )
    summary["shipping_cost_share"] = (
        summary["total_shipping_cost"] / summary["total_sales"]
    )

    figure = px.scatter(
        summary,
        x="average_completion_days",
        y="shipping_cost_share",
        size="total_shipping_cost",
        color="market",
        hover_name="market",
        hover_data={
            "total_orders": ":,",
            "total_sales": ":$,.2f",
            "total_shipping_cost": ":$,.2f",
            "average_shipping_cost_per_order": ":$,.2f",
        },
        title=title,
        labels={
            "average_completion_days": "Average Completion Time (Days)",
            "shipping_cost_share": "Shipping Cost as a Share of Sales",
        },
        template="plotly_white",
    )
    figure.update_yaxes(tickformat=".1%")
    return finish_dashboard_figure(figure)

# 9. Create filter options

Every dropdown includes an `All` option. This gives the dashboard a clear unfiltered starting point.

The years are stored as integers in the data. The labels shown to users are text, while each year's dropdown value remains an integer.

In [54]:
year_options = [
    {"label": "All Years", "value": "All"}
] + [
    {"label": str(year), "value": int(year)}
    for year in sorted(df_clean["year"].dropna().unique())
]

market_options = [
    {"label": "All Markets", "value": "All"}
] + [
    {"label": market, "value": market}
    for market in sorted(df_clean["market"].dropna().unique())
]

category_options = [
    {"label": "All Categories", "value": "All"}
] + [
    {"label": category, "value": category}
    for category in sorted(df_clean["category"].dropna().unique())
]

print("Year choices:", len(year_options))
print("Market choices:", len(market_options))
print("Category choices:", len(category_options))

Year choices: 5
Market choices: 8
Category choices: 4


# 10. Create the Dash layout

The layout defines what the user sees. It does not yet define what should happen when a filter changes.

Our layout contains:

1. A title and scope statement
2. Three dropdown filters and a reset button
3. Five headline KPI cards
4. Six dashboard charts
5. A short methodology note

Every interactive component has a unique `id`. The callbacks will use those IDs to connect inputs to outputs.

In [55]:
app = Dash(__name__)
app.title = "Global Superstore Dashboard"

app.layout = html.Div(
    [
        # Dashboard heading
        html.Div(
            [
                html.H1(
                    "Global Superstore Performance Dashboard",
                    style={
                        "margin": "0",
                        "color": "white",
                        "fontSize": "32px",
                    },
                ),
                html.P(
                    "Retained sales, profitability, returns and shipping performance",
                    style={
                        "margin": "8px 0 0 0",
                        "color": "#DCEAF7",
                        "fontSize": "15px",
                    },
                ),
            ],
            style={
                "background": (
                    f"linear-gradient(120deg, {COLORS['primary']}, "
                    f"{COLORS['secondary']})"
                ),
                "padding": "28px 32px",
                "borderRadius": "0 0 18px 18px",
            },
        ),

        # Filters
        html.Div(
            [
                html.Div(
                    [
                        html.Label(
                            "Year",
                            style={"fontWeight": "600", "fontSize": "13px"},
                        ),
                        dcc.Dropdown(
                            id="year-filter",
                            options=year_options,
                            value="All",
                            clearable=False,
                        ),
                    ],
                    style=FILTER_STYLE,
                ),
                html.Div(
                    [
                        html.Label(
                            "Market",
                            style={"fontWeight": "600", "fontSize": "13px"},
                        ),
                        dcc.Dropdown(
                            id="market-filter",
                            options=market_options,
                            value="All",
                            clearable=False,
                        ),
                    ],
                    style=FILTER_STYLE,
                ),
                html.Div(
                    [
                        html.Label(
                            "Product Category",
                            style={"fontWeight": "600", "fontSize": "13px"},
                        ),
                        dcc.Dropdown(
                            id="category-filter",
                            options=category_options,
                            value="All",
                            clearable=False,
                        ),
                    ],
                    style=FILTER_STYLE,
                ),
                html.Button(
                    "Reset Filters",
                    id="reset-filters-button",
                    n_clicks=0,
                    style={
                        "height": "38px",
                        "alignSelf": "end",
                        "padding": "0 18px",
                        "backgroundColor": COLORS["primary"],
                        "color": "white",
                        "border": "none",
                        "borderRadius": "7px",
                        "fontWeight": "600",
                        "cursor": "pointer",
                    },
                ),
            ],
            style={
                **CARD_STYLE,
                "display": "flex",
                "gap": "16px",
                "alignItems": "end",
                "flexWrap": "wrap",
                "margin": "22px 24px 12px 24px",
            },
        ),

        html.P(
            id="filter-summary",
            children=(
                f"Showing all {len(df_clean):,} order lines and "
                f"{df_clean['order_key'].nunique():,} distinct orders."
            ),
            style={
                "margin": "0 28px 14px 28px",
                "color": COLORS["muted"],
                "fontSize": "13px",
            },
        ),

        # Headline KPI cards
        html.Div(
            [
                make_kpi_card(
                    "Retained Sales",
                    "retained-sales-value",
                    "Sales from orders not listed as returned",
                    COLORS["secondary"],
                ),
                make_kpi_card(
                    "Retained Profit",
                    "retained-profit-value",
                    "Profit from orders not listed as returned",
                    COLORS["success"],
                ),
                make_kpi_card(
                    "Profit Margin",
                    "profit-margin-value",
                    "Retained profit divided by retained sales",
                    COLORS["success"],
                ),
                make_kpi_card(
                    "Retained Orders",
                    "retained-orders-value",
                    "Distinct non-returned market-order keys",
                    COLORS["primary"],
                ),
                make_kpi_card(
                    "Return Rate",
                    "return-rate-value",
                    "Returned distinct orders divided by all orders",
                    COLORS["accent"],
                ),
            ],
            style={
                "display": "grid",
                "gridTemplateColumns": "repeat(auto-fit, minmax(180px, 1fr))",
                "gap": "14px",
                "margin": "0 24px 20px 24px",
            },
        ),

        # Charts. The initial figures are replaced when the callback runs.
        html.Div(
            [
                html.Div(
                    dcc.Graph(
                        id="monthly-sales-chart",
                        figure=fig_monthly_sales,
                        config={"displaylogo": False},
                    ),
                    style=GRAPH_CARD_STYLE,
                ),
                html.Div(
                    dcc.Graph(
                        id="market-sales-chart",
                        figure=fig_market_sales,
                        config={"displaylogo": False},
                    ),
                    style=GRAPH_CARD_STYLE,
                ),
                html.Div(
                    dcc.Graph(
                        id="category-sales-chart",
                        figure=fig_category_sales,
                        config={"displaylogo": False},
                    ),
                    style=GRAPH_CARD_STYLE,
                ),
                html.Div(
                    dcc.Graph(
                        id="market-return-chart",
                        figure=fig_market_returns,
                        config={"displaylogo": False},
                    ),
                    style=GRAPH_CARD_STYLE,
                ),
                html.Div(
                    dcc.Graph(
                        id="subcategory-profit-chart",
                        figure=fig_subcategory_profit,
                        config={"displaylogo": False},
                    ),
                    style=GRAPH_CARD_STYLE,
                ),
                html.Div(
                    dcc.Graph(
                        id="shipping-chart",
                        figure=fig_shipping,
                        config={"displaylogo": False},
                    ),
                    style=GRAPH_CARD_STYLE,
                ),
            ],
            style={
                "display": "grid",
                "gridTemplateColumns": "repeat(auto-fit, minmax(430px, 1fr))",
                "gap": "16px",
                "margin": "0 24px 20px 24px",
            },
        ),

        # Methodology note
        html.Div(
            [
                html.H4(
                    "How to read this dashboard",
                    style={"margin": "0 0 8px 0"},
                ),
                html.P(
                    "Retained metrics exclude orders listed in the Returns sheet. "
                    "The return flag is available only at order level, so returned-order "
                    "sales represent original recorded sales associated with those orders—not "
                    "confirmed refund amounts.",
                    style={
                        "margin": "0",
                        "color": COLORS["muted"],
                        "fontSize": "13px",
                        "lineHeight": "1.6",
                    },
                ),
            ],
            style={
                **CARD_STYLE,
                "margin": "0 24px 30px 24px",
            },
        ),
    ],
    style={
        "backgroundColor": COLORS["page"],
        "minHeight": "100vh",
        "fontFamily": "Arial, sans-serif",
        "color": COLORS["text"],
    },
)

# 11. Create the dashboard callbacks

## Main dashboard callback

The three dropdown `value` properties are the inputs. When any value changes, Dash calls `update_dashboard()` automatically.

The function:

1. Filters `df_clean`.
2. Recalculates all five KPIs.
3. Rebuilds all six Plotly figures.
4. Returns twelve outputs in the exact order listed in the decorator.

One callback is appropriate here because all outputs depend on the same three filters and the same filtered dataset.

In [56]:
@app.callback(
    Output("retained-sales-value", "children"),
    Output("retained-profit-value", "children"),
    Output("profit-margin-value", "children"),
    Output("retained-orders-value", "children"),
    Output("return-rate-value", "children"),
    Output("monthly-sales-chart", "figure"),
    Output("market-sales-chart", "figure"),
    Output("category-sales-chart", "figure"),
    Output("market-return-chart", "figure"),
    Output("subcategory-profit-chart", "figure"),
    Output("shipping-chart", "figure"),
    Output("filter-summary", "children"),
    Input("year-filter", "value"),
    Input("market-filter", "value"),
    Input("category-filter", "value"),
)
def update_dashboard(selected_year, selected_market, selected_category):
    filtered_data = filter_dashboard_data(
        df_clean,
        selected_year,
        selected_market,
        selected_category,
    )

    kpis = calculate_dashboard_kpis(filtered_data)

    monthly_figure = build_monthly_sales_figure(filtered_data)
    market_figure = build_market_sales_figure(filtered_data)
    category_figure = build_category_sales_figure(filtered_data)
    return_figure = build_market_return_figure(filtered_data)
    profit_figure = build_subcategory_profit_figure(filtered_data)
    shipping_figure = build_shipping_figure(filtered_data)

    filter_message = (
        f"Showing {len(filtered_data):,} order lines and "
        f"{kpis['gross_orders']:,} distinct orders for the selected filters."
    )

    return (
        format_currency(kpis["retained_sales"]),
        format_currency(kpis["retained_profit"]),
        format_percentage(kpis["profit_margin"]),
        format_integer(kpis["retained_orders"]),
        format_percentage(kpis["return_rate"]),
        monthly_figure,
        market_figure,
        category_figure,
        return_figure,
        profit_figure,
        shipping_figure,
        filter_message,
    )

## Reset-filter callback

Clicking the reset button returns all three dropdowns to `"All"`. Changing those dropdown values then triggers the main callback automatically.

In [57]:
@app.callback(
    Output("year-filter", "value"),
    Output("market-filter", "value"),
    Output("category-filter", "value"),
    Input("reset-filters-button", "n_clicks"),
    prevent_initial_call=True,
)
def reset_filters(number_of_clicks):
    return "All", "All", "All"

## Test the callback calculations before starting the server

Dash normally calls the main callback when the application loads. We can also call the underlying Python function directly to verify that it returns the expected number and types of outputs.

In [58]:
test_outputs = update_dashboard("All", "All", "All")

assert len(test_outputs) == 12
assert test_outputs[0] == format_currency(total_sales)
assert test_outputs[1] == format_currency(total_profit)
assert test_outputs[2] == format_percentage(overall_profit_margin)
assert test_outputs[3] == format_integer(total_orders)
assert test_outputs[4] == format_percentage(return_rate)
assert all(hasattr(figure, "to_dict") for figure in test_outputs[5:11])

print("The unfiltered callback results match the validated pandas KPIs.")

The unfiltered callback results match the validated pandas KPIs.


# Run the completed dashboard

Dash 2.11 and later can run directly inside Jupyter. `jupyter_mode="external"` displays a link that opens the application in a browser tab.

Run the next cell only after all earlier cells have completed successfully.

> The cell remains active while the dashboard server is running. Use Jupyter's stop button or interrupt the kernel when you are finished.

In [59]:
app.run(
    jupyter_mode="external",
    debug=False,
    port=8050,
)

Dash app running on http://127.0.0.1:8050/


## If port 8050 is already in use

Stop the earlier Dash process or change the final line to another port:

```python
app.run(jupyter_mode="external", debug=False, port=8051)
```

If your environment does not recognize `jupyter_mode`, confirm that Dash 2.11 or later is installed.